In [ ]:
import numpy as np
    
def compute_iou_numpy(box, boxes):
    """
    Calculates the IoU between a single box and an array of boxes.
    
    Args:
        box (np.ndarray): A single box of shape (4,) representing [x1, y1, x2, y2]
        boxes (np.ndarray): Multiple boxes of shape (N, 4) representing N boxes
        
    Returns:
        np.ndarray: IoU values of shape (N,)
    """
    # 1. Find the coordinates of the intersecting rectangle
    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])

    # 2. Calculate intersection area (clamp negative width/height to 0)
    inter_width = np.maximum(0, x2 - x1)
    inter_height = np.maximum(0, y2 - y1)
    inter_area = inter_width * inter_height

    # 3. Calculate areas of the individual boxes
    box_area = (box[2] - box[0]) * (box[3] - box[1])
    boxes_area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])

    # 4. Calculate Union Area
    union_area = box_area + boxes_area - inter_area

    # 5. Compute IoU (adding a tiny epsilon to prevent division by zero)
    iou = inter_area / (union_area + 1e-6)
    
    return iou

In [ ]:
def nms_numpy(predictions, iou_threshold=0.5, conf_threshold=0.5):
    """
    Performs Non-Maximum Suppression on raw predictions.
    
    Args:
        predictions (np.ndarray): Shape (N, 6) format [class_id, score, x1, y1, x2, y2]
        iou_threshold (float): Threshold to drop overlapping boxes
        conf_threshold (float): Threshold to drop low-confidence boxes
        
    Returns:
        np.ndarray: The filtered boxes remaining after NMS
    """
    # 1. Filter out all boxes below the confidence threshold immediately
    conf_mask = predictions[:, 1] >= conf_threshold
    predictions = predictions[conf_mask]

    # If no boxes passed the threshold, return an empty array
    if len(predictions) == 0:
        return np.array([])

    # 2. Sort the boxes descending by confidence score
    # predictions[:, 1] targets the score column
    sort_indices = np.argsort(predictions[:, 1])[::-1]
    predictions = predictions[sort_indices]

    kept_boxes = []

    while len(predictions) > 0:
        # 3. Always keep the box with the highest score
        best_box = predictions[0]
        kept_boxes.append(best_box)

        # If it was the last box, we are done
        if len(predictions) == 1:
            break

        # Extract the rest of the boxes to compare against the best box
        rest_boxes = predictions[1:]

        # 4. Calculate IoU between the best box and ALL remaining boxes simultaneously
        # We pass only the coordinate columns [2:]
        ious = compute_iou_numpy(best_box[2:], rest_boxes[:, 2:])

        # 5. Create a Boolean Mask to filter the remaining boxes
        # Keep it IF it's a different class
        different_class = rest_boxes[:, 0] != best_box[0]
        # OR Keep it IF the IoU is strictly lower than our overlap threshold
        low_overlap = ious < iou_threshold
        
        # Combine conditions: Keep if either condition is True
        keep_mask = different_class | low_overlap

        # 6. Apply the mask to drop the highly overlapping boxes of the same class
        predictions = rest_boxes[keep_mask]

    return np.array(kept_boxes)


In [ ]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolo26n.pt")  # load ab pretrained model (recommended for training)

# Train the model with MPS
results = model.train(data="coco8.yaml", epochs=100, imgsz=640, device="mps", )

New https://pypi.org/project/ultralytics/8.4.66 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.13.13 torch-2.12.0 MPS (Apple M1 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-10, nbs=64, nms=False, opset=

In [8]:
model = YOLO("Users/rafael/projects/ml_bootcamp/current/classwork/runs/detect/train-10/weights/best.pt") 

FileNotFoundError: [Errno 2] No such file or directory: 'Users/rafael/projects/ml_bootcamp/current/classwork/runs/detect/train-10/weights/best.pt'

In [3]:
model

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=